In [1]:
from typing import List
from pydantic import BaseModel
from guardrails import Guard, settings

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="guardrails")

In [3]:
settings.disable_tracing = True
settings.rc.enable_metrics = False

In [4]:
class MovieReview(BaseModel):
    title: str
    sentiment: str  # 'positive' or 'negative'
    key_points: List[str]

In [5]:
guard = Guard.for_pydantic(output_class=MovieReview)

In [6]:
# Now validate model output
raw_output = """
{
  "title": "Inception",
  "sentiment": "positive",
  "key_points": ["Mind-bending plot", "Brilliant direction"]
}
"""


In [7]:
validated_output = guard.parse(raw_output)

In [8]:
validated_output.validation_passed

True

In [9]:
if validated_output.validation_passed:
    print("Validation Passed!")
    print(validated_output.validated_output)
else:
    print("Validation Failed!")
    print("Reason:", validated_output.reask.fail_results[0].error_message)

Validation Passed!
{'title': 'Inception', 'sentiment': 'positive', 'key_points': ['Mind-bending plot', 'Brilliant direction']}


In [10]:
raw_output = '''
{
  "title": "Inception",
  "key_points": ["Mind-bending plot", "Brilliant direction"]
}
'''

In [11]:
validated_output = guard.parse(raw_output)

In [12]:
validated_output.validation_passed

False

In [13]:
if validated_output.validation_passed:
    print("Validation Passed!")
    print(validated_output.validated_output)
else:
    print("Validation Failed!")
    print("Reason:", validated_output.reask.fail_results[0].error_message)

Validation Failed!
Reason: JSON does not match schema:
{
  "$": [
    "'sentiment' is a required property"
  ]
}


In [14]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Read OpenAI API Key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [16]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

In [17]:
prompt = """
Generate a structured JSON response for a movie review with the following keys:
- title: name of the movie
- sentiment: 'positive' or 'negative'
- key_points: a list of 2–3 bullet points summarizing the movie

Movie: Inception
"""

In [18]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that always responds in valid JSON only."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

In [19]:
response

ChatCompletion(id='chatcmpl-EfRa7RRlzKisZ4oE75kUM8hJhibHA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n  "title": "Inception",\n  "sentiment": "positive",\n  "key_points": [\n    "The film features a complex and intriguing plot that explores the concept of dreams within dreams.",\n    "Exceptional performances by the cast, particularly Leonardo DiCaprio, enhance the depth of the story.",\n    "Christopher Nolan\'s direction and the stunning visual effects create an immersive cinematic experience."\n  ]\n}\n```', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1793546179, model='gpt-4o-2024-08-06', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_3bfbcbf18d', usage=CompletionUsage(completion_tokens=87, prompt_tokens=80, total_tokens=167, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, au

In [20]:
generated_output = response.choices[0].message.content

In [21]:
generated_output

'```json\n{\n  "title": "Inception",\n  "sentiment": "positive",\n  "key_points": [\n    "The film features a complex and intriguing plot that explores the concept of dreams within dreams.",\n    "Exceptional performances by the cast, particularly Leonardo DiCaprio, enhance the depth of the story.",\n    "Christopher Nolan\'s direction and the stunning visual effects create an immersive cinematic experience."\n  ]\n}\n```'

'```json\n{\n  "title": "Inception",\n  "sentiment": "positive",\n  "key_points": [\n    "The film features a complex and intriguing plot that explores the concept of dreams within dreams.",\n    "Exceptional performances by the cast, particularly Leonardo DiCaprio, enhance the depth of the story.",\n    "Christopher Nolan\'s direction and the stunning visual effects create an immersive cinematic experience."\n  ]\n}\n```'

In [22]:
validated_output = guard.parse(generated_output)

In [23]:
validated_output.validation_passed

True

In [24]:
if validated_output.validation_passed:
    print("Validation Passed!")
    print(validated_output.validated_output)
else:
    print("Validation Failed!")
    print("Reason:", validated_output.reask.fail_results[0].error_message)

Validation Passed!
{'title': 'Inception', 'sentiment': 'positive', 'key_points': ['The film features a complex and intriguing plot that explores the concept of dreams within dreams.', 'Exceptional performances by the cast, particularly Leonardo DiCaprio, enhance the depth of the story.', "Christopher Nolan's direction and the stunning visual effects create an immersive cinematic experience."]}
